In [1]:
from pathlib import Path
import os, sys

PROJECT_ROOT = Path.cwd().resolve().parents[1]
if Path.cwd().resolve() != PROJECT_ROOT:
    os.chdir(PROJECT_ROOT)

root_str = str(PROJECT_ROOT)
if root_str in sys.path:
    sys.path.remove(root_str)
sys.path.insert(0, root_str)

#print("PROJECT_ROOT =", PROJECT_ROOT)
#print("cwd =", Path.cwd())

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import scanpy as sc
import pandas as pd
import numpy as np
import os
import torch

## input

In [4]:
from src.preprocessing import pp
from sklearn.model_selection import train_test_split
import scvi

In [7]:
control_key = "is_control"
condition_rep_keys = "perturbation_embeddings"
condition_combined_keys = "condition_combined"
mass_deduct_keys = None
random_seed = 42

condition_keys = "perturbation" # 数据集perturbation所对应的obs列名
dataset_name = "Replogle_essential_e5test1"
sample_rep = "X_pca" #"X_scVI"  "X_flatvi" "X_state"

cov_config = {
    # "batch": {
    #     "type": "categorical",
    #     "control_ot": "global",
    #     "perturbed_ot": "global",
    #     "use_in_model": True,
    #     "model_source": "control",
    #     "contain_in_condition": False,
    #     "condition_source": None,
    # },
}


if_adata_ref = None #用于根据一个参考adata快速构建pca
adata_ref_path = "data/processed/PBMC2000_pca_rep_0.2_42.h5ad"

condition_rep_dict = pd.read_pickle("./data/processed/condition_embedding_gene_replogle_essential_e5test1.pkl")
condition_rep_dict = {
    k: (v["embedding"] if isinstance(v, dict) and "embedding" in v else None)
    for k, v in condition_rep_dict.items()
}

In [8]:
filePath = './data/raw/Replogle_hvg2000.h5ad'
adata = sc.read_h5ad(filePath)
adata.uns["cov_config"] = cov_config
print(adata)

AnnData object with n_obs × n_vars = 310385 × 2000
    obs: 'batch', 'gene', 'gene_id', 'transcript', 'gene_transcript', 'guide_id', 'percent_mito', 'UMI_count', 'z_gemgroup_UMI', 'core_scale_factor', 'core_adjusted_UMI_count', 'disease', 'cancer', 'cell_line', 'sex', 'age', 'perturbation', 'organism', 'perturbation_type', 'tissue_type', 'ncounts', 'ngenes', 'nperts', 'percent_ribo'
    var: 'chr', 'start', 'end', 'class', 'strand', 'length', 'in_matrix', 'mean', 'std', 'cv', 'fano', 'ensembl_id', 'ncounts', 'ncells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'hvg', 'log1p', 'cov_config'
    layers: 'counts'


In [9]:
adata.obs[control_key] = (adata.obs[condition_keys] == "control")
condition_list = adata[adata.obs[control_key]==False].obs[condition_keys].unique()
print(adata.obs[control_key].value_counts())

is_control
False    299694
True      10691
Name: count, dtype: int64


## splitting

In [10]:
rng = np.random.default_rng(random_seed) 
test_ratio = 0.2
condition_list = list(condition_list)
zero_shot = True

if not zero_shot:
    # 分层抽样 先验证分布内学习能力
    pert_mask = adata.obs[control_key] == False
    y = adata.obs.loc[pert_mask, condition_keys].astype(str).values
    pert_indices = np.flatnonzero(pert_mask)

    train_idx, test_idx = train_test_split(
        pert_indices,
        test_size=test_ratio,
        random_state=random_seed,
        stratify=y 
    )
    adata_train = adata[train_idx].copy()
    adata_test = adata[test_idx].copy()
    adata_control = adata[adata.obs[control_key] == True].copy()
    adata_train.uns["normalized_m"] = 1 / (1-test_ratio)
    adata_test.uns["normalized_m"] = 1 / test_ratio
    adata_control.uns["normalized_m"] = 1 
    print(condition_list)
else:
    # 按condition分割 zeroshot
    n_test = max(1, int(len(condition_list) * test_ratio))
    test_condition = rng.choice(condition_list, size=n_test, replace=False).tolist()
    print(test_condition)
    train_condition = [g for g in condition_list if g not in test_condition]
    print(train_condition)
    adata_control = adata[adata.obs[control_key]==True].copy() # control的target_condition是 PBS
    adata_train = adata[adata.obs[condition_keys].isin(train_condition)].copy() 
    adata_test = adata[adata.obs[condition_keys].isin(test_condition)].copy()
    adata_train.uns["normalized_m"] = 1
    adata_test.uns["normalized_m"] = 1
    adata_control.uns["normalized_m"] = 1 

['NDUFB3', 'UTP20', 'TEAD3', 'PSMB7', 'SRSF11', 'SRP14', 'PSMG3', 'SF3B2', 'GPN2', 'TUBGCP2', 'CADM4', 'ZNHIT6', 'ZNRD1', 'UBA3', 'RNF8', 'RPN2', 'OR4N2', 'TANGO6', 'RPL34', 'INTS6', 'LTV1', 'ZCCHC9', 'CYC1', 'ANAPC11', 'DNM2', 'TRAPPC8', 'POGZ', 'NXF1', 'ZNF236', 'QRSL1', 'NBPF12', 'MYC', 'CDT1', 'EXOC7', 'CD3EAP', 'SNRNP25', 'HDAC3', 'RUVBL1', 'TTI1', 'TOP1', 'TMED2', 'KIN', 'VPS28', 'PCBP2', 'TRAPPC1', 'DOHH', 'RBBP8', 'USP19', 'DNLZ', 'TOP3A', 'HARS2', 'PRPF38A', 'GMPS', 'MCM2', 'MRPS22', 'RAB18', 'RPS14', 'NUFIP1', 'MBTPS2', 'PWP1', 'PSMD6', 'PSMD14', 'FBLIM1', 'SCAF1', 'HNRNPH1', 'BMS1', 'GSDMA', 'HSP90B1', 'VPS13D', 'CDC7', 'EXOSC4', 'FAM136A', 'SART1', 'GRSF1', 'RCOR1', 'CEP85', 'DTL', 'TAF6', 'RPL30', 'PCNA', 'RPS21', 'RFC4', 'ESPN', 'INO80', 'PRKRA', 'STRAP', 'RPSA', 'KIF18A', 'TFIP11', 'PMF1', 'SARS2', 'RPS17', 'ATL2', 'SNAPC4', 'PRPF6', 'SRSF1', 'RPL19', 'CLP1', 'RPL12', 'NDC80', 'RBM19', 'CLK2', 'RNF103', 'DYNC1I2', 'CHMP7', 'TRAPPC11', 'MRPS35', 'ZBTB17', 'TTF2', 'TBCE', 

In [11]:
del adata

## latent embedding

In [12]:
n_comps = 100
n_hidden = 1024
n_layers = 2
#condition_rep_dict = pd.read_pickle("./data/processed/PBMC_cytokines.pkl")
model_ref = None
model_train = None
model_test = None
scvi_save_path = f"./data/processed/model/{sample_rep}_ncomps{n_comps}_hidden{n_hidden}_layers{n_layers}_{dataset_name}_{random_seed}_{test_ratio}_{zero_shot}"
flatvi_save_path = f"./data/processed/model/{sample_rep}_ncomps{n_comps}_hidden{n_hidden}_layers{n_layers}_{dataset_name}"
state_decoder_save_path = f"./data/processed/model/{sample_rep}_{dataset_name}"
model_save_path = None

load_embedding_model = True
if sample_rep in ["X_scVI"]:
    model_save_path = scvi_save_path
    if load_embedding_model:
        try:
            model_ref = scvi.model.SCVI.load(f"{model_save_path}_ref", adata=adata_control)
        except (FileNotFoundError, OSError, ValueError) as e:
            model_ref = None 
        try:
            model_train = scvi.model.SCVI.load(f"{model_save_path}_train", adata=adata_train)
        except (FileNotFoundError, OSError, ValueError) as e:
            model_train = None
        if adata_test is not None:
            try:
                model_test = scvi.model.SCVI.load(f"{model_save_path}_test", adata=adata_test)
            except (FileNotFoundError, OSError, ValueError) as e:
                model_test = None
elif sample_rep == "X_flatvi":
    model_save_path = flatvi_save_path
    if load_embedding_model:
        try:
            model_ref = scvi.model.SCVI.load(f"{flatvi_save_path}_ref", adata=adata_control)
        except (FileNotFoundError, OSError, ValueError) as e:
            model_ref = None 
elif sample_rep=="X_state":
    from src.preprocessing import build_train_eval_loaders,NBDecoderTrainer,NBDecoder
    model_save_path = state_decoder_save_path
    z_dim = adata_control.obsm["X_state"].shape[1] # 2058
    n_genes = adata_control.n_vars
    decoder = NBDecoder(z_dim=z_dim, n_genes=n_genes, hidden=(1024,2048,4096), dropout=0.1)
    train_loader, val_loader = build_train_eval_loaders(
                                    adata_train=adata_control,
                                    adata_eval=adata_train,   
                                    count_layer="counts",
                                    emb_key="X_state",
                                    batch_size=256,
                                )
    trainer = NBDecoderTrainer(decoder, lr=1e-4, device="cuda", use_amp=True)
    trainer.fit(train_loader, val_loader=val_loader, epochs=50)
    trainer.save(f"{state_decoder_save_path}.pt")

In [13]:
if if_adata_ref:
    adata_ref = sc.read_h5ad(adata_ref_path,backed='r')
else:
    adata_ref = None

In [14]:
adata_control, adata_train, adata_test, model_ref, model_train, model_test = pp.process_to_embedding( 
    adata_control,
    adata_train,
    adata_ref = adata_ref,
    adata_test = adata_test,
    sample_rep = sample_rep,
    n_comps = n_comps,
    n_hidden = n_hidden,
    n_layers = n_layers,
    model_ref = model_ref,
    model_train = model_train,
    model_test = model_test,
    model_save_path = model_save_path,
    control_key = control_key,
    condition_keys = condition_keys,
    condition_rep_keys = condition_rep_keys,
    condition_combined_keys = condition_combined_keys,
    cov_config = cov_config,
    condition_rep_dict = condition_rep_dict,
    pca_method = "scanpy", # "parse"
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    )

[2.2623935  2.367287   1.9139715  1.8297055  1.81917    1.1978222
 1.3383542  1.0753545  1.0328404  1.0290146  1.0127153  0.95647883
 0.93538046 0.8071712  0.8608224  0.8179949  0.75224304 0.7707254
 0.7213388  0.72942626 0.72734916 0.68339103 0.688414   0.6803144
 0.68638194 0.67538416 0.6505124  0.6587757  0.6140806  0.62263423
 0.61816686 0.6196176  0.6189738  0.6245067  0.6021523  0.62003696
 0.60092425 0.56319046 0.57722896 0.56085986 0.5756992  0.56429666
 0.56679475 0.56718254 0.55995    0.5648367  0.5503813  0.54271066
 0.5257108  0.54842424 0.54089713 0.53868484 0.5326373  0.5282808
 0.52730745 0.52389944 0.516206   0.5243282  0.51410884 0.5078969
 0.51690286 0.5055922  0.49645704 0.5013146  0.50868386 0.49838167
 0.48677036 0.5004559  0.49640867 0.49381408 0.48169225 0.4870114
 0.47922188 0.47781563 0.48285285 0.47804323 0.47346616 0.4749055
 0.47986963 0.47995505 0.48069412 0.4772668  0.47666717 0.4737165
 0.4676274  0.46600714 0.46381336 0.4607054  0.45979866 0.45921135
 0.

In [15]:
preprocess_save_path = f"./data/processed/{dataset_name}_{random_seed}_{test_ratio}_{zero_shot}_{sample_rep}_{n_comps}_{if_adata_ref}"
adata_control.write_h5ad(f"{preprocess_save_path}_control.h5ad")
adata_train.write_h5ad(f"{preprocess_save_path}_train.h5ad")
if adata_test is not None:
    adata_test.write_h5ad(f"{preprocess_save_path}_test.h5ad")

In [16]:
print(preprocess_save_path)
print(adata_control)
print(adata_train)
print(adata_test)

./data/processed/Replogle_essential_e5test1_42_0.2_True_X_pca_100_None
AnnData object with n_obs × n_vars = 10691 × 2000
    obs: 'batch', 'gene', 'gene_id', 'transcript', 'gene_transcript', 'guide_id', 'percent_mito', 'UMI_count', 'z_gemgroup_UMI', 'core_scale_factor', 'core_adjusted_UMI_count', 'disease', 'cancer', 'cell_line', 'sex', 'age', 'perturbation', 'organism', 'perturbation_type', 'tissue_type', 'ncounts', 'ngenes', 'nperts', 'percent_ribo', 'is_control', 'condition_combined'
    var: 'chr', 'start', 'end', 'class', 'strand', 'length', 'in_matrix', 'mean', 'std', 'cv', 'fano', 'ensembl_id', 'ncounts', 'ncells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'hvg', 'log1p', 'cov_config', 'normalized_m', 'pca', 'global_rulebook'
    obsm: 'X_pca'
    varm: 'PCs', 'X_mean'
    layers: 'counts'
AnnData object with n_obs × n_vars = 241026 × 2000
    obs: 'batch', 'gene', 'gene_id', 'transcript', 'gene_transcript', 'guide_id', 'percent_mito', 'UMI_count', '

In [ ]:
adata_control.uns

In [ ]:
adata_train.obs[condition_combined_keys].value_counts()

In [ ]:
denoised_df = model_ref.get_normalized_expression(adata_control, return_mean=True,library_size=1e4)
raw_adata = adata_control.copy()
sc.pp.normalize_total(raw_adata, target_sum=1e4)
raw_matrix = raw_adata.X
if hasattr(raw_matrix, "toarray"):
    raw_matrix = raw_matrix.toarray()

In [ ]:
# 计算原始数据和重建数据的基因均值
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
mean_raw = np.mean(raw_matrix, axis=0)
mean_recon = np.mean(denoised_df.values, axis=0)

# 计算相关性 (Pearson 或 Spearman)
corr, _ = pearsonr(mean_raw, mean_recon)
print(f"Gene Mean Correlation (Raw vs Recon): {corr:.4f}")

# 可视化
plt.figure(figsize=(6, 6))
plt.scatter(mean_raw, mean_recon, s=1, alpha=0.5)
plt.plot([0, max(mean_raw)], [0, max(mean_raw)], 'r--') # 对角线
plt.xlabel("Raw Mean Expression (Normalized)")
plt.ylabel("Reconstructed Mean Expression")
plt.title(f"Reconstruction Quality (R = {corr:.2f})")
plt.xscale('log')
plt.yscale('log')
plt.show()